# 05: Batch Scan — All 30 Cases (24 DC-AC + 6 DC Baselines)

## Goal

Run the full Phase 0→3 protocol for **all 30 cases** in the master table:
- **24 DC-AC cases**: I_DC ∈ {0.1, 0.2, 0.3, 0.5, 0.7}C × multiple A_Crate × multiple frequencies
- **6 DC baselines**: I_DC ∈ {0.1, 0.2, 0.3, 0.5, 0.7, 0.9}C, A=0, f=0 (degenerate to pure CC-CV)

Both case types share **the same code path**:

    I(t) = I_DC + A · sin(2π · f · t)

For DC baseline, `A=0` reduces this to `I(t) = I_DC` (constant). For batch consistency, all cases use `pybamm.step.CustomStepExplicit`.

## Architecture

`run_single_case(I_DC_Crate, A_Crate, f_Hz, ...) → results_dict` encapsulates Cells 3-10b of `04_dcac_single_case.ipynb`. Output is a list of 30 dicts, archived to JSON for Day 7 figure generation.

## Strict-net constraints (carried over from Day 5)

1. Signed integration (no |I|, no rectification)
2. Allow non-monotonic Q_net(t)
3. First-passage only
4. Unified Q-grid across cases (applied at post-processing, not here)
5. No signal modification, no AC smoothing
6. Δt(Q) is a curve, not a scalar (Day 7 deliverable)

## Validation

The `0.3C+0.7C @ 10τ` case from Day 5 is re-run via the function and compared against Day 5 output to verify equivalence (CC time should match within < 1 s).

In [1]:
# ============================================================
# Cell 2: Imports + run_single_case function
# ============================================================
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid
import time

print(f"PyBaMM version: {pybamm.__version__}")
print(f"NumPy version:  {np.__version__}")


def run_single_case(
    I_DC_Crate: float,
    A_Crate: float,
    f_Hz: float,
    parameter_set: str = "Chen2020",
    T_ambient_C: float = 20.0,
    nominal_capacity_Ah: float = 5.0,
    t_phase2_start_s: float = 8118.5,
    verbose: bool = False,
) -> dict:
    """
    Run Phase 0a→0b→1→2→3 protocol for a single DC-AC (or DC baseline) case.

    Parameters
    ----------
    I_DC_Crate : float
        DC base C-rate. Charging direction is enforced internally (sign flipped to negative).
        Pass the absolute C-rate (e.g. 0.3 for 0.3C charging).
    A_Crate : float
        AC amplitude in C-rate units. Set 0 for DC baseline.
    f_Hz : float
        AC frequency in Hz. Ignored when A_Crate == 0.
    parameter_set : str
        PyBaMM parameter set name (default Chen2020).
    T_ambient_C : float
        Ambient temperature in °C (default 20).
    nominal_capacity_Ah : float
        Nominal capacity for C-rate → A conversion (default 5.0 Ah for Chen2020).
    t_phase2_start_s : float
        Absolute simulation time at which Phase 2 starts. Constant for shared Phase 0+1
        protocol; default 8118.5 s from Day 5 calibration.
    verbose : bool
        Print per-phase diagnostics if True.

    Returns
    -------
    dict
        - case identifiers: case_id, I_DC_Crate, A_Crate, f_Hz, kappa
        - event layer: CC_time_s, CV_time_s, total_time_s (and _min versions)
        - state layer: Q_net_final_mAh, Q_net_trajectory, t_chg, I_chg
        - thermal: T_max_charging_C
        - strict-net stats: n_AC_reversals_steps, pct_AC_reversals
        - meta: wall_clock_s, status ("ok" or error string)
    """
    # ------------- Sanitize inputs -------------
    # Handle NaN / None for A_Crate (DC baseline rows in CSV)
    if A_Crate is None or (isinstance(A_Crate, float) and np.isnan(A_Crate)):
        A_Crate = 0.0
    if f_Hz is None or (isinstance(f_Hz, float) and np.isnan(f_Hz)):
        f_Hz = 0.0

    # Convert C-rates to absolute currents (PyBaMM sign: + = discharge, − = charge)
    I_DC_A = -abs(I_DC_Crate) * nominal_capacity_Ah   # always charging
    A_A    = A_Crate * nominal_capacity_Ah

    # Avoid kappa division-by-zero for DC baseline
    kappa = (A_Crate / abs(I_DC_Crate)) if abs(I_DC_Crate) > 1e-9 else 0.0

    # case_id naming
    if A_Crate < 1e-9:
        case_id = f"DC{abs(I_DC_Crate):.2f}C_baseline"
    else:
        # 10τ ≈ 0.00143 Hz; we don't try to back-compute τ-multiple here, just use Hz
        case_id = f"DC{abs(I_DC_Crate):.2f}C+AC{A_Crate:.2f}C_f{f_Hz:.5f}Hz"

    # ------------- Define DC-AC current function -------------
    def dc_ac_current(variables):
        t_absolute = variables["Time [s]"]
        t_relative = t_absolute - t_phase2_start_s
        return I_DC_A + A_A * pybamm.sin(2 * np.pi * f_Hz * t_relative)

    # ------------- Build experiment -------------
    dcac_step = pybamm.step.CustomStepExplicit(
        dc_ac_current,
        termination="4.2V",
        direction="charge",
        description=f"Phase 2: {case_id}",
    )

    experiment = pybamm.Experiment([
        "Discharge at 1C until 2.5V",
        "Hold at 2.5V until C/136",
        "Rest for 3805 seconds",
        dcac_step,
        "Hold at 4.2V until C/68",
    ])

    # ------------- Build model + run -------------
    model = pybamm.lithium_ion.SPMe(options={"thermal": "lumped"})
    pv = pybamm.ParameterValues(parameter_set)
    pv["Ambient temperature [K]"] = T_ambient_C + 273.15
    pv["Initial temperature [K]"] = T_ambient_C + 273.15

    sim = pybamm.Simulation(model, parameter_values=pv, experiment=experiment)

    t_start = time.time()
    try:
        solution = sim.solve()
        wall_clock_s = time.time() - t_start
        status = "ok"
    except Exception as e:
        wall_clock_s = time.time() - t_start
        return {
            "case_id": case_id, "I_DC_Crate": I_DC_Crate, "A_Crate": A_Crate, "f_Hz": f_Hz,
            "kappa": kappa, "wall_clock_s": wall_clock_s, "status": f"FAILED: {type(e).__name__}: {e}",
        }

    # ------------- Extract per-phase data -------------
    phase2 = solution.cycles[3]
    phase3 = solution.cycles[4]

    cc_time_s = phase2.t[-1] - phase2.t[0]
    cv_time_s = phase3.t[-1] - phase3.t[0]
    total_time_s = cc_time_s + cv_time_s

    # Concatenate Phase 2 + 3 for strict-net Q(t)
    t_chg = np.concatenate([phase2.t, phase3.t])
    I_chg = np.concatenate([phase2["Current [A]"].entries, phase3["Current [A]"].entries])
    _, unique_idx = np.unique(t_chg, return_index=True)
    unique_idx_sorted = np.sort(unique_idx)
    t_chg = t_chg[unique_idx_sorted]
    I_chg = I_chg[unique_idx_sorted]
    t_chg = t_chg - t_chg[0]   # re-reference to Phase 2 start

    # Strict-net signed integration: Q_net = -∫I dt (since I<0 = charge)
    Q_net_As = -cumulative_trapezoid(I_chg, t_chg, initial=0)
    Q_net_mAh = Q_net_As / 3600.0 * 1000.0
    Q_final_mAh = float(Q_net_mAh[-1])

    # Non-monotonicity stats
    dQ = np.diff(Q_net_mAh)
    n_decreases = int(np.sum(dQ < 0))
    pct_decreases = 100.0 * n_decreases / max(len(dQ), 1)

    # Thermal
    T_max_p2 = float(phase2["X-averaged cell temperature [K]"].entries.max() - 273.15)
    T_max_p3 = float(phase3["X-averaged cell temperature [K]"].entries.max() - 273.15)
    T_max_charging = max(T_max_p2, T_max_p3)

    if verbose:
        print(f"  [{case_id}] CC={cc_time_s/60:.2f}min, CV={cv_time_s/60:.2f}min, "
              f"T_max={T_max_charging:.2f}°C, Q_net={Q_final_mAh:.0f}mAh, "
              f"AC reversal={pct_decreases:.1f}%, wall={wall_clock_s:.2f}s")

    return {
        # Identifiers
        "case_id":           case_id,
        "I_DC_Crate":        abs(I_DC_Crate),     # report as positive C-rate
        "A_Crate":           A_Crate,
        "f_Hz":              f_Hz,
        "kappa":             kappa,
        # Event layer
        "CC_time_s":         cc_time_s,
        "CV_time_s":         cv_time_s,
        "total_time_s":      total_time_s,
        "CC_time_min":       cc_time_s / 60,
        "CV_time_min":       cv_time_s / 60,
        "total_time_min":    total_time_s / 60,
        # State layer
        "Q_net_final_mAh":   Q_final_mAh,
        "Q_net_trajectory":  Q_net_mAh.tolist(),  # JSON-serializable
        "t_chg":             t_chg.tolist(),
        "I_chg":             I_chg.tolist(),
        # Thermal
        "T_max_charging_C":  T_max_charging,
        # Strict-net stats
        "n_AC_reversals_steps": n_decreases,
        "pct_AC_reversals":  pct_decreases,
        # Meta
        "wall_clock_s":      wall_clock_s,
        "status":            status,
    }

print("✓ run_single_case defined.")

PyBaMM version: 26.3.1
NumPy version:  2.4.4
✓ run_single_case defined.


In [2]:
# ============================================================
# Cell 3: Equivalence test — re-run Day 5's representative case
# Expected: CC time = 9569.5 s, Q_net = 5127 mAh, T_max = 26.78°C
# ============================================================

print("Running 0.3C+0.7C @ 10τ (Day 5 representative case)...")
print()

day5_case = run_single_case(
    I_DC_Crate=0.3,
    A_Crate=0.7,
    f_Hz=0.00143,
    verbose=True,
)

# Day 5 reference values (from Cell 10b output)
day5_reference = {
    "CC_time_s":       9569.5,
    "CV_time_s":       4554.7,
    "Q_net_final_mAh": 5127,
    "T_max_charging_C": 26.78,
    "pct_AC_reversals": 32.1,
}

print()
print("=" * 78)
print(f"{'Metric':<28} {'Function':>14} {'Day 5':>14} {'Δ':>20}")
print("-" * 78)

for k, ref_v in day5_reference.items():
    func_v = day5_case[k]
    delta = func_v - ref_v
    if k == "Q_net_final_mAh":
        print(f"{k:<28} {func_v:>14.0f} {ref_v:>14.0f} {delta:>+10.1f} mAh")
    elif "time" in k:
        print(f"{k:<28} {func_v:>10.1f} s   {ref_v:>10.1f} s   {delta:>+10.2f} s")
    elif k == "T_max_charging_C":
        print(f"{k:<28} {func_v:>14.3f} {ref_v:>14.3f} {delta:>+10.3f} °C")
    elif "pct" in k:
        print(f"{k:<28} {func_v:>14.2f} {ref_v:>14.2f} {delta:>+10.2f} %")

print("-" * 78)
print(f"Wall clock: {day5_case['wall_clock_s']:.2f} s")
print(f"Status: {day5_case['status']}")
print("=" * 78)

# Hard equivalence check
tolerances = {
    "CC_time_s":        1.0,    # 1 second
    "Q_net_final_mAh":  10.0,   # 10 mAh
    "T_max_charging_C": 0.1,    # 0.1 °C
}

all_pass = True
for k, tol in tolerances.items():
    diff = abs(day5_case[k] - day5_reference[k])
    if diff > tol:
        print(f"⚠ FAIL: {k} differs by {diff:.3f} (tolerance {tol})")
        all_pass = False

if all_pass:
    print("✓ All key metrics match Day 5 within tolerances. Function is equivalent.")
else:
    print("✗ Equivalence FAILED. Investigate before batch run.")

Running 0.3C+0.7C @ 10τ (Day 5 representative case)...

  [DC0.30C+AC0.70C_f0.00143Hz] CC=159.49min, CV=75.91min, T_max=26.78°C, Q_net=5127mAh, AC reversal=32.1%, wall=0.78s

Metric                             Function          Day 5                    Δ
------------------------------------------------------------------------------
CC_time_s                        9569.5 s       9569.5 s        -0.01 s
CV_time_s                        4554.7 s       4554.7 s        -0.04 s
Q_net_final_mAh                        5127           5127       +0.0 mAh
T_max_charging_C                     26.781         26.780     +0.001 °C
pct_AC_reversals                      32.07          32.10      -0.03 %
------------------------------------------------------------------------------
Wall clock: 0.78 s
Status: ok
✓ All key metrics match Day 5 within tolerances. Function is equivalent.


In [3]:
# ============================================================
# Cell 4: DC baseline test — verify A=0 degenerates to pure DC
# Compare against Day 4's 0.9C DC baseline (CC=49.48 min experimental, 48.91 min sim)
# ============================================================

print("Running 0.9C DC baseline (A=0)...")
print()

dc_baseline_case = run_single_case(
    I_DC_Crate=0.9,
    A_Crate=0.0,        # pure DC
    f_Hz=0.0,           # ignored when A=0
    verbose=True,
)

# Day 4 reference (from Day 4 commit e4de1a3)
day4_reference = {
    "CC_time_min":       48.91,
    "CV_time_min":       85.74,
    "Q_net_final_mAh":   5135,
    "T_max_charging_C":  31.52,
}

print()
print("=" * 78)
print(f"{'Metric':<28} {'Function':>14} {'Day 4':>14} {'Δ':>20}")
print("-" * 78)
for k, ref_v in day4_reference.items():
    func_v = dc_baseline_case[k]
    delta = func_v - ref_v
    print(f"{k:<28} {func_v:>14.2f} {ref_v:>14.2f} {delta:>+10.3f}")
print("-" * 78)
print(f"Wall clock: {dc_baseline_case['wall_clock_s']:.2f} s")
print(f"Status: {dc_baseline_case['status']}")
print(f"AC reversal % (should be ~0 for DC): {dc_baseline_case['pct_AC_reversals']:.2f}%")
print(f"kappa (should be 0 for DC): {dc_baseline_case['kappa']}")
print("=" * 78)

# Sanity check: AC reversal should be 0 (or very close, due to numerical noise)
if dc_baseline_case['pct_AC_reversals'] < 1.0:
    print("✓ AC reversal ~0% as expected for pure DC.")
else:
    print(f"⚠ AC reversal = {dc_baseline_case['pct_AC_reversals']:.2f}% — unexpected for DC baseline.")

if dc_baseline_case['kappa'] == 0.0:
    print("✓ kappa = 0 as expected for DC baseline.")

Running 0.9C DC baseline (A=0)...

  [DC0.90C_baseline] CC=48.90min, CV=78.12min, T_max=31.52°C, Q_net=5126mAh, AC reversal=0.0%, wall=0.61s

Metric                             Function          Day 4                    Δ
------------------------------------------------------------------------------
CC_time_min                           48.90          48.91     -0.011
CV_time_min                           78.12          85.74     -7.621
Q_net_final_mAh                     5126.31        5135.00     -8.689
T_max_charging_C                      31.52          31.52     -0.001
------------------------------------------------------------------------------
Wall clock: 0.61 s
Status: ok
AC reversal % (should be ~0 for DC): 0.00%
kappa (should be 0 for DC): 0.0
✓ AC reversal ~0% as expected for pure DC.
✓ kappa = 0 as expected for DC baseline.


In [4]:
# ============================================================
# Cell 5: Read master CSV + sanity check
# ============================================================
import pandas as pd

CSV_PATH = "../data/figure1_master_table_cleaned.csv"

df = pd.read_csv(CSV_PATH)
print(f"CSV loaded: {len(df)} rows, {len(df.columns)} columns")
print()

# Show relevant columns only
relevant_cols = ["DC_base", "Condition", "f_Hz", "DC_C", "AC_C", "I_DC_A", "A_A", 
                 "kappa", "T_max", "Delta_t_Q80_min", "total_time_min"]
print("Relevant columns preview:")
print(df[relevant_cols].to_string(max_rows=10))
print()

# Verify DC baseline vs DC-AC split
n_baseline = (df["AC_C"] == 0).sum()
n_dcac = (df["AC_C"] != 0).sum()
print(f"DC baseline cases: {n_baseline}")
print(f"DC-AC cases:       {n_dcac}")
print(f"Total:             {len(df)}")

# Check for any unexpected NaN in critical columns
for col in ["DC_C", "AC_C", "f_Hz"]:
    nan_count = df[col].isna().sum()
    if nan_count > 0:
        print(f"⚠ {col} has {nan_count} NaN values")
    else:
        print(f"✓ {col}: no NaN")

CSV loaded: 30 rows, 26 columns

Relevant columns preview:
   DC_base       Condition      f_Hz  DC_C  AC_C  I_DC_A   A_A  kappa   T_max  Delta_t_Q80_min  total_time_min
0     0.1C     0.1+0.9C 1τ  0.014300   0.1   0.9    0.34  3.06    9.0  26.376             2.88      523.416667
1     0.1C     0.1+0.2C 1τ  0.014300   0.1   0.2    0.34  0.68    2.0  23.059             0.59      593.533333
2     0.2C     0.2+0.3C 1τ  0.014300   0.2   0.3    0.68  1.02    1.5  24.323             0.17      307.650000
3     0.2C    0.2+0.3C 10τ  0.001430   0.2   0.3    0.68  1.02    1.5  24.503             5.05      298.700000
4     0.2C  0.2+0.3C 34.8τ  0.000412   0.2   0.3    0.68  1.02    1.5  24.994            15.03      306.550000
..     ...             ...       ...   ...   ...     ...   ...    ...     ...              ...             ...
25    0.2C         0.2C_DC  0.000000   0.2   0.0    0.68   NaN    NaN  22.769              NaN             NaN
26    0.3C         0.3C_DC  0.000000   0.3   0.0    1

In [8]:
# ============================================================
# Cell 6: Batch scan all 30 cases
# Wall-clock estimate: ~30 case × 0.7s ≈ 25-40 s total
# ============================================================
import time

results_list = []
total_start = time.time()

print(f"Scanning {len(df)} cases...\n")

for i, row in df.iterrows():
    print(f"[{i+1:2d}/{len(df)}] {row['Condition']:<25}", end="  ", flush=True)
    
    case_start = time.time()
    result = run_single_case(
        I_DC_Crate = float(row["DC_C"]),
        A_Crate    = float(row["AC_C"]),  # 0 for DC baseline
        f_Hz       = float(row["f_Hz"]),  # 0 for DC baseline (degenerate, doesn't matter)
        verbose    = False,
    )
    
    # Attach experimental ground truth from CSV for direct comparison
    result["exp_T_max_C"]            = float(row["T_max"]) if pd.notna(row["T_max"]) else None
    result["exp_Delta_t_Q80_min"]    = float(row["Delta_t_Q80_min"]) if pd.notna(row["Delta_t_Q80_min"]) else None
    result["exp_total_time_min"]     = float(row["total_time_min"]) if pd.notna(row["total_time_min"]) else None
    result["condition_label"]        = row["Condition"]
    result["DC_base_label"]          = row["DC_base"]
    
    results_list.append(result)
    
    case_elapsed = time.time() - case_start
    if result["status"] == "ok":
        print(f"CC={result['CC_time_min']:6.2f}min  T_max={result['T_max_charging_C']:5.2f}°C  "
              f"Q={result['Q_net_final_mAh']:.0f}mAh  wall={case_elapsed:.2f}s")
    else:
        print(f"FAILED: {result['status']}")

total_elapsed = time.time() - total_start
n_ok       = sum(1 for r in results_list if r["status"] == "ok")
n_failed   = len(results_list) - n_ok

print()
print("=" * 78)
print(f"Total wall-clock: {total_elapsed:.1f} s")
print(f"Successful cases: {n_ok}/{len(results_list)}")
print(f"Failed cases:     {n_failed}/{len(results_list)}")
print(f"Avg per case:     {total_elapsed/len(results_list):.2f} s")

Scanning 30 cases...

[ 1/30] 0.1+0.9C 1τ                CC=498.57min  T_max=26.63°C  Q=5126mAh  wall=1.15s
[ 2/30] 0.1+0.2C 1τ                CC=583.60min  T_max=20.79°C  Q=5126mAh  wall=0.84s
[ 3/30] 0.2+0.3C 1τ                CC=277.09min  T_max=21.87°C  Q=5126mAh  wall=0.85s
[ 4/30] 0.2+0.3C 10τ               CC=276.00min  T_max=22.24°C  Q=5126mAh  wall=0.65s
[ 5/30] 0.2+0.3C 34.8τ             CC=272.70min  T_max=23.21°C  Q=5127mAh  wall=0.71s
[ 6/30] 0.2+0.8C 0.1τ              CC=248.57min  T_max=25.35°C  Q=5127mAh  wall=1.48s
[ 7/30] 0.2+0.8C 1τ                CC=244.48min  T_max=25.55°C  Q=5127mAh  wall=0.92s
[ 8/30] 0.2+0.8C 10τ               

2026-04-25 13:01:40.827 - [WARNING] callbacks.on_experiment_infeasible_event(254): 

	Experiment is infeasible: 'event: Minimum voltage [V]' was triggered during 'Phase 2: DC0.20C+AC0.80C_f0.00143Hz'. The returned solution only contains up to step 1 of cycle 4. 


FAILED: INFEASIBLE_MIN_V: V_min=1.500V in Phase 2 (Chen2020 more sensitive than MJ1)
[ 9/30] 0.3+0.7C 0.1τ              CC=162.21min  T_max=25.07°C  Q=5126mAh  wall=1.26s
[10/30] 0.3+0.7C 1τ                CC=159.43min  T_max=25.33°C  Q=5128mAh  wall=0.76s
[11/30] 0.3+0.7C 10τ               CC=159.49min  T_max=26.78°C  Q=5127mAh  wall=0.72s
[12/30] 0.3+0.4C 0.1τ              CC=169.66min  T_max=23.52°C  Q=5127mAh  wall=1.11s
[13/30] 0.3+0.4C 1τ                CC=168.68min  T_max=23.61°C  Q=5127mAh  wall=0.79s
[14/30] 0.3+0.4C 10τ               CC=170.75min  T_max=23.82°C  Q=5127mAh  wall=0.65s
[15/30] 0.4+0.6C 0.5τ              CC=118.73min  T_max=25.41°C  Q=5125mAh  wall=0.83s
[16/30] 0.4+0.5C 1.67τ             CC=119.60min  T_max=25.07°C  Q=5128mAh  wall=0.68s
[17/30] 0.4+0.6C 1.67τ             CC=117.68min  T_max=25.64°C  Q=5127mAh  wall=0.70s
[18/30] 0.4+0.6C 5τ                CC=114.95min  T_max=26.22°C  Q=5126mAh  wall=0.75s
[19/30] 0.4+0.6C 10τ               CC=113.54min  T_max=

In [10]:
# ============================================================
# Cell 7 (PATCHED): Archive results + tabular sanity check
# Handles infeasible cases with .get() fallbacks
# ============================================================
import json
from pathlib import Path

# Save full results to JSON (including trajectories — for Day 7 Δt(Q) computation)
RESULTS_PATH = Path("../data/results_day6_batch_scan.json")
with open(RESULTS_PATH, "w") as f:
    json.dump(results_list, f, indent=2)

print(f"Results archived: {RESULTS_PATH}")
print(f"File size: {RESULTS_PATH.stat().st_size / 1024:.1f} KB")
print()

# Build summary DataFrame — gracefully handle infeasible cases
summary_rows = []
for r in results_list:
    summary_rows.append({
        "condition":      r.get("condition_label", r["case_id"]),
        "I_DC_C":         r["I_DC_Crate"],
        "A_C":            r["A_Crate"],
        "f_Hz":           r["f_Hz"],
        "kappa":          r["kappa"],
        "sim_CC_min":     r.get("CC_time_min"),
        "sim_CV_min":     r.get("CV_time_min"),
        "sim_total_min":  r.get("total_time_min"),
        "sim_T_max_C":    r.get("T_max_charging_C") or r.get("T_max_charging_C_partial"),
        "sim_Q_mAh":      r.get("Q_net_final_mAh"),
        "exp_T_max_C":    r.get("exp_T_max_C"),
        "exp_Δt_Q80":     r.get("exp_Delta_t_Q80_min"),
        "exp_total_min":  r.get("exp_total_time_min"),
        "AC_rev_pct":     r.get("pct_AC_reversals"),
        "wall_s":         r.get("wall_clock_s"),
        "status":         r["status"],
    })

summary_df = pd.DataFrame(summary_rows)

# Save lightweight summary to CSV (no trajectories — quick to inspect)
SUMMARY_CSV = Path("../data/results_day6_summary.csv")
summary_df.to_csv(SUMMARY_CSV, index=False, float_format="%.3f")
print(f"Summary archived: {SUMMARY_CSV}")
print()

# Print summary, status grouped
print("=" * 110)
print("=== Successful cases (status='ok') ===")
print("=" * 110)
ok_df = summary_df[summary_df["status"] == "ok"].copy()
print(ok_df.drop(columns=["status"]).to_string(max_rows=30, float_format=lambda x: f"{x:.2f}" if isinstance(x, float) else x))
print()

if (summary_df["status"] != "ok").any():
    print("=" * 110)
    print("=== Infeasible / failed cases ===")
    print("=" * 110)
    fail_df = summary_df[summary_df["status"] != "ok"]
    print(fail_df[["condition", "I_DC_C", "A_C", "f_Hz", "kappa", "status"]].to_string())
    print()

# Sim vs exp T_max comparison (only for cases with both)
print("=" * 110)
print("=== Sim vs Exp T_max (where both available) ===")
print("=" * 110)
both = summary_df[(summary_df["sim_T_max_C"].notna()) & (summary_df["exp_T_max_C"].notna())].copy()
both["ΔT_max"] = both["sim_T_max_C"] - both["exp_T_max_C"]
print(both[["condition", "kappa", "sim_T_max_C", "exp_T_max_C", "ΔT_max"]].to_string(
    max_rows=30, float_format=lambda x: f"{x:.2f}" if isinstance(x, float) else x))
print()
print(f"Mean ΔT_max (sim − exp): {both['ΔT_max'].mean():+.2f}°C")
print(f"Std  ΔT_max:             {both['ΔT_max'].std():.2f}°C")

Results archived: ../data/results_day6_batch_scan.json
File size: 9869.3 KB

Summary archived: ../data/results_day6_summary.csv

=== Successful cases (status='ok') ===
         condition  I_DC_C  A_C  f_Hz  kappa  sim_CC_min  sim_CV_min  sim_total_min  sim_T_max_C  sim_Q_mAh  exp_T_max_C  exp_Δt_Q80  exp_total_min  AC_rev_pct  wall_s
0      0.1+0.9C 1τ    0.10 0.90  0.01   9.00      498.57       69.83         568.40        26.63    5125.68        26.38        2.88         523.42       46.32    1.08
1      0.1+0.2C 1τ    0.10 0.20  0.01   2.00      583.60       47.07         630.67        20.79    5126.21        23.06        0.59         593.53       32.72    0.82
2      0.2+0.3C 1τ    0.20 0.30  0.01   1.50      277.09       58.55         335.64        21.87    5126.45        24.32        0.17         307.65       25.88    0.75
3     0.2+0.3C 10τ    0.20 0.30  0.00   1.50      276.00       61.08         337.08        22.24    5125.87        24.50        5.05         298.70       25.06 

In [7]:
# ============================================================
# Cell 8: Patched run_single_case with infeasibility handling
# Replaces Cell 2's function. Same signature, same return schema.
# ============================================================

def run_single_case(
    I_DC_Crate: float,
    A_Crate: float,
    f_Hz: float,
    parameter_set: str = "Chen2020",
    T_ambient_C: float = 20.0,
    nominal_capacity_Ah: float = 5.0,
    t_phase2_start_s: float = 8118.5,
    verbose: bool = False,
) -> dict:
    """Run Phase 0a→0b→1→2→3 protocol. Handles event-triggered infeasibility."""
    
    # Sanitize inputs
    if A_Crate is None or (isinstance(A_Crate, float) and np.isnan(A_Crate)):
        A_Crate = 0.0
    if f_Hz is None or (isinstance(f_Hz, float) and np.isnan(f_Hz)):
        f_Hz = 0.0
    
    I_DC_A = -abs(I_DC_Crate) * nominal_capacity_Ah
    A_A    = A_Crate * nominal_capacity_Ah
    kappa  = (A_Crate / abs(I_DC_Crate)) if abs(I_DC_Crate) > 1e-9 else 0.0
    
    if A_Crate < 1e-9:
        case_id = f"DC{abs(I_DC_Crate):.2f}C_baseline"
    else:
        case_id = f"DC{abs(I_DC_Crate):.2f}C+AC{A_Crate:.2f}C_f{f_Hz:.5f}Hz"
    
    # Build current function
    def dc_ac_current(variables):
        t_absolute = variables["Time [s]"]
        t_relative = t_absolute - t_phase2_start_s
        return I_DC_A + A_A * pybamm.sin(2 * np.pi * f_Hz * t_relative)
    
    dcac_step = pybamm.step.CustomStepExplicit(
        dc_ac_current, termination="4.2V", direction="charge",
        description=f"Phase 2: {case_id}",
    )
    experiment = pybamm.Experiment([
        "Discharge at 1C until 2.5V",
        "Hold at 2.5V until C/136",
        "Rest for 3805 seconds",
        dcac_step,
        "Hold at 4.2V until C/68",
    ])
    
    model = pybamm.lithium_ion.SPMe(options={"thermal": "lumped"})
    pv = pybamm.ParameterValues(parameter_set)
    pv["Ambient temperature [K]"] = T_ambient_C + 273.15
    pv["Initial temperature [K]"] = T_ambient_C + 273.15
    
    sim = pybamm.Simulation(model, parameter_values=pv, experiment=experiment)
    
    t_start = time.time()
    try:
        solution = sim.solve()
    except Exception as e:
        return {
            "case_id": case_id, "I_DC_Crate": abs(I_DC_Crate), "A_Crate": A_Crate, "f_Hz": f_Hz,
            "kappa": kappa, "wall_clock_s": time.time() - t_start,
            "status": f"FAILED_SOLVE: {type(e).__name__}: {e}",
        }
    wall_clock_s = time.time() - t_start
    
    # ----- NEW: detect infeasibility -----
    n_cycles = len(solution.cycles)
    expected_cycles = 5
    
    base_dict = {
        "case_id": case_id, "I_DC_Crate": abs(I_DC_Crate), "A_Crate": A_Crate, "f_Hz": f_Hz,
        "kappa": kappa, "wall_clock_s": wall_clock_s,
    }
    
    if n_cycles < 4:
        # Failed before Phase 2 even started — should not happen with this protocol
        return {**base_dict, "status": f"INFEASIBLE_EARLY: only {n_cycles} cycles completed"}
    
    if n_cycles < expected_cycles:
        # Phase 2 was infeasible (event triggered Min V) — extract partial Phase 2 data only
        phase2_partial = solution.cycles[3]
        cc_time_partial_s = phase2_partial.t[-1] - phase2_partial.t[0]
        T_max_partial = float(phase2_partial["X-averaged cell temperature [K]"].entries.max() - 273.15)
        V_min_phase2 = float(phase2_partial["Terminal voltage [V]"].entries.min())
        
        if verbose:
            print(f"  [{case_id}] INFEASIBLE: Phase 2 hit Min V ({V_min_phase2:.3f}V). "
                  f"Partial CC={cc_time_partial_s/60:.2f}min")
        
        return {
            **base_dict,
            "status": f"INFEASIBLE_MIN_V: V_min={V_min_phase2:.3f}V in Phase 2 (Chen2020 more sensitive than MJ1)",
            "CC_time_s_partial": cc_time_partial_s,
            "T_max_charging_C_partial": T_max_partial,
            "V_min_phase2": V_min_phase2,
        }
    
    # ----- Normal case (5 cycles complete) -----
    phase2 = solution.cycles[3]
    phase3 = solution.cycles[4]
    
    cc_time_s = phase2.t[-1] - phase2.t[0]
    cv_time_s = phase3.t[-1] - phase3.t[0]
    total_time_s = cc_time_s + cv_time_s
    
    t_chg = np.concatenate([phase2.t, phase3.t])
    I_chg = np.concatenate([phase2["Current [A]"].entries, phase3["Current [A]"].entries])
    _, unique_idx = np.unique(t_chg, return_index=True)
    unique_idx_sorted = np.sort(unique_idx)
    t_chg = t_chg[unique_idx_sorted]
    I_chg = I_chg[unique_idx_sorted]
    t_chg = t_chg - t_chg[0]
    
    Q_net_As = -cumulative_trapezoid(I_chg, t_chg, initial=0)
    Q_net_mAh = Q_net_As / 3600.0 * 1000.0
    Q_final_mAh = float(Q_net_mAh[-1])
    
    dQ = np.diff(Q_net_mAh)
    n_decreases = int(np.sum(dQ < 0))
    pct_decreases = 100.0 * n_decreases / max(len(dQ), 1)
    
    T_max_p2 = float(phase2["X-averaged cell temperature [K]"].entries.max() - 273.15)
    T_max_p3 = float(phase3["X-averaged cell temperature [K]"].entries.max() - 273.15)
    T_max_charging = max(T_max_p2, T_max_p3)
    
    if verbose:
        print(f"  [{case_id}] CC={cc_time_s/60:.2f}min, CV={cv_time_s/60:.2f}min, "
              f"T_max={T_max_charging:.2f}°C, Q_net={Q_final_mAh:.0f}mAh, "
              f"AC reversal={pct_decreases:.1f}%, wall={wall_clock_s:.2f}s")
    
    return {
        **base_dict,
        "CC_time_s": cc_time_s, "CV_time_s": cv_time_s, "total_time_s": total_time_s,
        "CC_time_min": cc_time_s/60, "CV_time_min": cv_time_s/60, "total_time_min": total_time_s/60,
        "Q_net_final_mAh": Q_final_mAh,
        "Q_net_trajectory": Q_net_mAh.tolist(),
        "t_chg": t_chg.tolist(),
        "I_chg": I_chg.tolist(),
        "T_max_charging_C": T_max_charging,
        "n_AC_reversals_steps": n_decreases,
        "pct_AC_reversals": pct_decreases,
        "status": "ok",
    }

print("✓ run_single_case patched — now handles infeasible cases gracefully.")

✓ run_single_case patched — now handles infeasible cases gracefully.


In [11]:
# ============================================================
# Cell 8.5: Honest T_max statistics (exclude infeasible cases)
# ============================================================

# Filter: only successful cases with both sim and exp T_max
clean_df = summary_df[
    (summary_df["status"] == "ok") 
    & summary_df["sim_T_max_C"].notna() 
    & summary_df["exp_T_max_C"].notna()
].copy()
clean_df["ΔT_max"] = clean_df["sim_T_max_C"] - clean_df["exp_T_max_C"]

print(f"Clean dataset (ok + both T_max available): {len(clean_df)} cases")
print(f"Mean ΔT_max: {clean_df['ΔT_max'].mean():+.2f}°C")
print(f"Std  ΔT_max: {clean_df['ΔT_max'].std():.2f}°C")
print(f"Min  ΔT_max: {clean_df['ΔT_max'].min():+.2f}°C")
print(f"Max  ΔT_max: {clean_df['ΔT_max'].max():+.2f}°C")
print()

# Stratify by case type
dc_only = clean_df[clean_df["A_C"] == 0]
dcac    = clean_df[clean_df["A_C"] != 0]

print(f"Stratified:")
print(f"  DC baselines (n={len(dc_only)}): mean ΔT={dc_only['ΔT_max'].mean():+.2f}°C, std={dc_only['ΔT_max'].std():.2f}°C")
print(f"  DC-AC (n={len(dcac)}):           mean ΔT={dcac['ΔT_max'].mean():+.2f}°C, std={dcac['ΔT_max'].std():.2f}°C")
print()

# Stratify by kappa regime (DC-AC only)
print("DC-AC stratified by kappa:")
for kappa_lo, kappa_hi in [(0, 0.5), (0.5, 1.5), (1.5, 3), (3, 10)]:
    bin_df = dcac[(dcac["kappa"] > kappa_lo) & (dcac["kappa"] <= kappa_hi)]
    if len(bin_df) > 0:
        print(f"  κ ∈ ({kappa_lo:.1f}, {kappa_hi:.1f}], n={len(bin_df):2d}: "
              f"mean ΔT={bin_df['ΔT_max'].mean():+.2f}°C, std={bin_df['ΔT_max'].std():.2f}°C")

# Note infeasible
print()
infeasible_df = summary_df[summary_df["status"] != "ok"]
if len(infeasible_df) > 0:
    print(f"⚠ {len(infeasible_df)} infeasible case(s) excluded from T_max statistics:")
    for _, r in infeasible_df.iterrows():
        print(f"  - {r['condition']}: {r['status']}")

Clean dataset (ok + both T_max available): 29 cases
Mean ΔT_max: -1.90°C
Std  ΔT_max: 0.84°C
Min  ΔT_max: -3.53°C
Max  ΔT_max: +0.25°C

Stratified:
  DC baselines (n=6): mean ΔT=-2.36°C, std=0.61°C
  DC-AC (n=23):           mean ΔT=-1.77°C, std=0.86°C

DC-AC stratified by kappa:
  κ ∈ (0.0, 0.5], n= 4: mean ΔT=-2.05°C, std=0.70°C
  κ ∈ (0.5, 1.5], n=12: mean ΔT=-2.14°C, std=0.60°C
  κ ∈ (1.5, 3.0], n= 4: mean ΔT=-1.34°C, std=0.92°C
  κ ∈ (3.0, 10.0], n= 3: mean ΔT=-0.53°C, std=0.67°C

⚠ 1 infeasible case(s) excluded from T_max statistics:
  - 0.2+0.8C 10τ: INFEASIBLE_MIN_V: V_min=1.500V in Phase 2 (Chen2020 more sensitive than MJ1)


In [12]:
# Cell 8.6: Inspect highest-kappa bin
high_k = clean_df[clean_df["kappa"] > 3].sort_values("kappa", ascending=False)
print("κ > 3 bin (n=3 after excluding infeasible):")
print(high_k[["condition", "kappa", "f_Hz", "sim_CC_min", "sim_T_max_C", "exp_T_max_C", "ΔT_max"]].to_string())

κ > 3 bin (n=3 after excluding infeasible):
       condition  kappa    f_Hz  sim_CC_min  sim_T_max_C  exp_T_max_C    ΔT_max
0    0.1+0.9C 1τ    9.0  0.0143  498.566885    26.627105       26.376  0.251105
5  0.2+0.8C 0.1τ    4.0  0.1430  248.569661    25.351543       26.215 -0.863457
6    0.2+0.8C 1τ    4.0  0.0143  244.479062    25.553115       26.517 -0.963885


In [13]:
# Cell 8.7: DC baseline ΔT_max breakdown
dc_only_sorted = clean_df[clean_df["A_C"] == 0].sort_values("I_DC_C")
print("DC baseline ΔT_max:")
print(dc_only_sorted[["condition", "I_DC_C", "sim_T_max_C", "exp_T_max_C", "ΔT_max"]].to_string())

DC baseline ΔT_max:
   condition  I_DC_C  sim_T_max_C  exp_T_max_C    ΔT_max
24   0.1C_DC     0.1    20.316672       23.431 -3.114328
25   0.2C_DC     0.2    21.017789       22.769 -1.751211
26   0.3C_DC     0.3    22.050505       23.553 -1.502495
27   0.4C_DC     0.4    23.165783       25.867 -2.701217
28   0.5C_DC     0.5    24.648309       27.109 -2.460691
29   0.9C_DC     0.9    31.518918       34.170 -2.651082
